In [41]:
from datetime import date

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
import polars as pl
target_date = date(2026,3,1)
df_flights = get_bts_flights_for_day(target_date)

226 overlapping flights found for 2026-03-01


In [49]:
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.adsbx.readsb import read_adsbx_readsb
from flights.flights_match_adsb import get_matching_icaos_in_flights
df_adsb = read_adsbx_readsb()
expr = (pl.col("landing_time").dt.date() == target_date) & (pl.col("takeoff_time").dt.date() == target_date)
df_flights = df_flights.filter(expr)
df_matched = get_matching_icaos_in_flights(df_flights, df_adsb)

In [54]:
icao_category = (
    df_flights
    .select([
        "icao",
        pl.when(pl.col("category").is_null() | (pl.col("category") == ""))
        .then(pl.lit("unknown"))
        .otherwise(pl.col("category"))
        .alias("category"),
    ])
    .unique()
)

matched_icaos = df_matched.select("icao").unique()

category_match_pct = (
    icao_category
    .with_columns(pl.col("icao").is_in(matched_icaos.get_column("icao")).alias("perfect_match"))
    .group_by("category")
    .agg([
        pl.len().alias("total_icaos"),
        pl.col("perfect_match").sum().alias("perfect_match_icaos"),
    ])
    .with_columns(
        (pl.col("perfect_match_icaos") / pl.col("total_icaos") * 100)
        .round(2)
        .alias("perfect_match_pct")
    )
    .sort("category")
)

category_match_pct

/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_35118/4236042461.py:17: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .with_columns(pl.col("icao").is_in(matched_icaos.get_column("icao")).alias("perfect_match"))


category,total_icaos,perfect_match_icaos,perfect_match_pct
str,u32,u32,f64
"""A3""",4438,767,17.28


In [55]:
matched_icaos = matched_icaos["icao"].to_list()

In [56]:
icaos_that_dont_match = set(df_flights.get_column("icao").unique().to_list()) - set(matched_icaos)

In [57]:
display(len(df_flights.get_column("icao").unique()))
display(len(matched_icaos))
len(icaos_that_dont_match)

4438

767

3671

In [58]:
icao = list(icaos_that_dont_match)[4]

In [59]:
df_flights.filter(pl.col("icao") == icao)

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""ac62a9""",null,"""N8977G""",2026-03-01 01:14:00,"""KBNA""",2026-03-01 02:43:00,"""KDAL""",false,false,false,false,null,null,null,"""A3"""
"""ac62a9""",null,"""N8977G""",2026-03-01 03:40:00,"""KDAL""",2026-03-01 04:22:00,"""KLIT""",false,false,false,false,null,null,null,"""A3"""
"""ac62a9""",null,"""N8977G""",2026-03-01 11:50:00,"""KLIT""",2026-03-01 12:37:00,"""KSTL""",false,false,false,false,null,null,null,"""A3"""
"""ac62a9""",null,"""N8977G""",2026-03-01 14:19:00,"""KSTL""",2026-03-01 16:08:00,"""KMCO""",false,false,false,false,null,null,null,"""A3"""
"""ac62a9""",null,"""N8977G""",2026-03-01 17:35:00,"""KMCO""",2026-03-01 19:44:00,"""KIND""",false,false,false,false,null,null,null,"""A3"""
"""ac62a9""",null,"""N8977G""",2026-03-01 20:43:00,"""KIND""",2026-03-01 21:52:00,"""KATL""",false,false,false,false,null,null,null,"""A3"""


In [38]:
import os
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import requests

FR24_FLIGHT_SUMMARY_LIGHT_URL = (
    "https://fr24api.flightradar24.com/api/flight-summary/light"
)


def _format_fr24_datetime(value: datetime) -> str:
    """
    FR24 Flight Summary expects UTC like: YYYY-MM-DDTHH:MM:SSZ
    Treat naive datetimes as UTC.
    """
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    else:
        value = value.astimezone(timezone.utc)

    return value.strftime("%Y-%m-%dT%H:%M:%SZ")


def get_fr24_flight_summary_light_by_registration(
    *,
    flight_datetime_from: datetime,
    flight_datetime_to: datetime,
    registrations: str | list[str],
    api_key: str | None = None,
    limit: int = 20_000,
) -> pd.DataFrame:
    if isinstance(registrations, list):
        registrations_param = ",".join(reg.strip().upper() for reg in registrations)
    else:
        registrations_param = registrations.strip().upper()

    params = {
        "flight_datetime_from": _format_fr24_datetime(flight_datetime_from),
        "flight_datetime_to": _format_fr24_datetime(flight_datetime_to),
        "registrations": registrations_param,
        "limit": limit,
    }

    token = api_key or os.environ["FR24_API_KEY"]

    headers = {
        "Accept": "application/json",
        "Accept-Version": "v1",
        "Authorization": f"Bearer {token}",
    }

    response = requests.get(
        FR24_FLIGHT_SUMMARY_LIGHT_URL,
        params=params,
        headers=headers,
        timeout=30,
    )

    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        raise requests.HTTPError(
            f"{e}\n\nURL: {response.url}\n\nResponse body:\n{response.text}"
        ) from e

    payload: Any = response.json()

    if isinstance(payload, dict):
        rows = payload.get("data", [])
    elif isinstance(payload, list):
        rows = payload
    else:
        raise ValueError(f"Unexpected FR24 response shape: {type(payload)}")

    return pd.DataFrame(rows)


df = get_fr24_flight_summary_light_by_registration(
    flight_datetime_from=datetime(2026, 3, 1),
    flight_datetime_to=datetime(2026, 3, 2),
    registrations=["N419SF"],
)

df

,fr24_id,flight,callsign,operating_as,painted_as,type,reg,orig_icao,datetime_takeoff,dest_icao,dest_icao_actual,datetime_landed,hex,first_seen,last_seen,flight_ended
0,3e8d6457,None,N419SF,None,None,C172,N419SF,KLHZ,2026-03-01T18:19:54Z,None,None,None,A4F725,2026-03-01T18:21:58Z,2026-03-01T20:09:13Z,True


In [7]:
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
df_adsbx_flights = get_adsbx_flights_for_day(target_date)
expr = (pl.col("takeoff_time").dt.date() == target_date) & (pl.col("landing_time").dt.date() == target_date)

In [8]:
df_flights = df_flights.filter(expr)
df_adsbx_flights = df_adsbx_flights.filter(expr)

In [14]:
from flights_ai_model.validation import flights_df_comparison
df_adsbx_flights = df_adsbx_flights.filter(pl.col("icao").is_in(df_flights.get_column("icao")))
flights_df_comparison(df_adsbx_flights,df_flights)


--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 1796
Predicted flights: 3543  |  Gold flights: 3180
Correctly matched: 2361  |  Predicted unmatched: 1182  |  Gold flights missed: 819
Precision: 66.64%  |  Recall: 74.25%  |  F1: 70.24%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 2139/2361 matched (90.60%)
landing_airport_ident: 1870/2361 matched (79.20%)
overall: 4009/4722 matched (84.90%)


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_96918/2280423705.py:2: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_adsbx_flights = df_adsbx_flights.filter(pl.col("icao").is_in(df_flights.get_column("icao")))


(shape: (4_362, 28)
 ┌────────┬──────────┬────────────┬────────────┬───┬──────────┬────────────┬────────────┬───────────┐
 │ icao   ┆ callsign ┆ registrati ┆ takeoff_ti ┆ … ┆ category ┆ takeoff_ti ┆ landing_ti ┆ match_sta │
 │ ---    ┆ ---      ┆ on         ┆ me         ┆   ┆ ---      ┆ me_diff    ┆ me_diff    ┆ tus       │
 │ str    ┆ str      ┆ ---        ┆ ---        ┆   ┆ str      ┆ ---        ┆ ---        ┆ ---       │
 │        ┆          ┆ str        ┆ datetime[m ┆   ┆          ┆ duration[m ┆ duration[m ┆ str       │
 │        ┆          ┆            ┆ s]         ┆   ┆          ┆ s]         ┆ s]         ┆           │
 ╞════════╪══════════╪════════════╪════════════╪═══╪══════════╪════════════╪════════════╪═══════════╡
 │ a14096 ┆ null     ┆ N18ZD      ┆ 2026-03-01 ┆ … ┆          ┆ 1s         ┆ 2s         ┆ matched   │
 │        ┆          ┆            ┆ 01:22:01   ┆   ┆          ┆            ┆            ┆           │
 │ aa4b55 ┆ null     ┆ N7620G     ┆ 2026-03-01 ┆ … ┆ A1       